# POTOK: прогон тестов в Colab (ячейки Притока)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/COL-O/POTOK/blob/arena%2F2db7afdd-potok/pritok/colab/progon.ipynb)

Ноутбук собран **Притоком** (`Приток <pritok@potok.local>`) — не Сферой и не основателем.
Он ничего не решает и не меняет в доме: только **прогоняет** и печатает факт.

**Зачем он нужен.** В песочнице Arena нет Go (сеть до `go.dev`, `dl.google.com`,
`proxy.golang.org` закрыта), логи CI не читаются, артефакт CI лежит на закрытом
`blob.core.windows.net`. Поэтому единственный способ получить настоящий прогон —
ячейки, запущенные на живой машине основателя.

**Что делает ноутбук:**
1. клонирует ветку `arena/3bbbb51d-potok` (дом) и ветку сессии;
2. ставит свежий Go, печатает `go version` и версию дистрибутива;
3. подкладывает **пять патчей Притока** в отдельную копию дома и прогоняет
   `go build ./...` и `go test -race ./...` — сначала **без** патчей, потом **с** ними;
4. складывает полный лог в файл и печатает итог: что падает до, что после.

**Секретов не вставляет.** Токен не нужен: репозиторий публичный.


## 1. Клон и версия Go

In [ ]:
import os, subprocess, sys, textwrap

BRANCH_HOUSE = 'arena/3bbbb51d-potok'   # дом
BRANCH_PATCH = 'arena/2db7afdd-potok'   # ветка, где лежат патчи Притока
REPO = '/content/POTOK'

def sh(cmd, cwd=None, check=False):
    r = subprocess.run(cmd, shell=isinstance(cmd, str), cwd=cwd,
                       capture_output=True, text=True)
    if check and r.returncode != 0:
        print(r.stdout[-3000:]); print(r.stderr[-3000:])
        raise SystemExit(f'команда провалилась: {cmd}')
    return r

# Go из apt: на Ubuntu 22.04 это 1.18 (go.mod требует 1.21 — сборка откажется),
# на 24.04 — 1.22. Поэтому ставим официальный тарбол нужной версии.
sh('apt-get -qq update', check=True)
sh('apt-get -qq install -y curl git > /dev/null 2>&1', check=True)
GO_VER = '1.22.12'
if not os.path.exists('/usr/local/go/bin/go'):
    sh(f'curl -sSL -o /tmp/go.tgz https://go.dev/dl/go{GO_VER}.linux-amd64.tar.gz', check=True)
    sh('tar -C /usr/local -xzf /tmp/go.tgz', check=True)
os.environ['PATH'] = '/usr/local/go/bin:' + os.environ['PATH']

print(sh('cat /etc/os-release | head -2').stdout)
print(sh('go version').stdout.strip())
print(sh(f'gcc --version | head -1').stdout.strip() or 'gcc нет — -race не соберётся')

if not os.path.exists(REPO):
    sh(f'git clone -q https://github.com/COL-O/POTOK.git {REPO}', check=True)
sh('git fetch -q --all', cwd=REPO, check=True)
print(sh(f'git log --oneline -1 origin/{BRANCH_HOUSE}', cwd=REPO).stdout.strip())


## 2. Чистый прогон дома — без патчей

Это то, что должно совпасть с красным CI (`verify: failure`).
Лог печатается целиком в файл, в ячейке — только хвост.


In [ ]:
import shutil, glob

def run_suite(workdir, logpath, race=True):
    cmds = [['go','build','./...']]
    cmds.append(['go','test'] + (['-race'] if race else []) + ['./...'])
    with open(logpath, 'w') as f:
        for c in cmds:
            f.write(f'\n===== $ {" ".join(c)}\n')
            r = subprocess.run(c, cwd=workdir, capture_output=True, text=True,
                               env={**os.environ, 'CGO_ENABLED':'1'})
            f.write(r.stdout); f.write(r.stderr)
            f.write(f'===== код возврата: {r.returncode}\n')
    return logpath

def tail(path, n=2500):
    s = open(path, encoding='utf-8', errors='replace').read()
    return s[-n:]

CLEAN = '/content/house-clean'
shutil.rmtree(CLEAN, ignore_errors=True)
sh(f'git -c advice.detachedHead=false clone -q -b {BRANCH_HOUSE} https://github.com/COL-O/POTOK.git {CLEAN}', check=True)

log = run_suite(CLEAN, '/content/log-clean.txt')
print(tail(log))


## 3. С патчами Притока — те же команды

Патчи берутся из ветки `arena/2db7afdd-potok` (`pritok/patches/0001…0005`) и применяются
по порядку. Если какой-то не применился — ячейка это скажет и не будет врать «всё хорошо».


In [ ]:
PATCHED = '/content/house-patched'
shutil.rmtree(PATCHED, ignore_errors=True)
sh(f'git -c advice.detachedHead=false clone -q -b {BRANCH_HOUSE} https://github.com/COL-O/POTOK.git {PATCHED}', check=True)

names = ['0001-core-one-bus','0002-core-eventbus-unsubscribe','0003-seed-created-at',
         '0004-stratos-wal-durability','0005-links-real-wiring']
applied = []
for n in names:
    p = f'pritok/patches/{n}.patch'
    raw = f'https://raw.githubusercontent.com/COL-O/POTOK/{BRANCH_PATCH}/{p}'
    dst = f'/content/{n}.patch'
    r = sh(f'curl -sSL -o {dst} {raw}')
    chk = sh(f'git apply --check {dst}', cwd=PATCHED)
    if chk.returncode != 0:
        print(f'{n}: НЕ ПРИМЕНИЛСЯ\n{chk.stderr[:800]}')
        continue
    sh(f'git apply {dst}', cwd=PATCHED, check=True)
    applied.append(n)
print('применено патчей:', len(applied), 'из', len(names), applied)

log2 = run_suite(PATCHED, '/content/log-patched.txt')
print(tail(log2))


## 4. Итог: что именно падало до и что после

In [ ]:
import re

def failures(path):
    txt = open(path, encoding='utf-8', errors='replace').read()
    bad = sorted(set(re.findall(r'^--- FAIL: (\S+)', txt, re.M)))
    pkg = sorted(set(re.findall(r'^FAIL\s+(\S+)', txt, re.M)))
    return bad, pkg

for label, path in [('БЕЗ патчей','/content/log-clean.txt'),
                    ('С патчами','/content/log-patched.txt')]:
    if not os.path.exists(path):
        print(label, '— лога нет'); continue
    bad, pkg = failures(path)
    print(f'--- {label}')
    print('  пакеты с FAIL:', ', '.join(pkg) or 'нет')
    print('  тесты:', ', '.join(bad) or 'нет')

print()
print('Полные логи:', '/content/log-clean.txt', '/content/log-patched.txt')
print('Скачать: слева панель файлов -> правый клик -> Download.')


## 5. Что делать с результатом

- Прислать основателю **весь** `/content/log-patched.txt`, если он короткий, либо имена
  упавших тестов из ячейки 4 — этого достаточно, чтобы не гадать.
- Если с патчами падений меньше — это ещё **не** «зелено»: нужен полный прогон без падений.
- Если падений столько же — патчи не лечат причину, и это тоже факт: тогда правим по логу.

**Чего этот ноутбук не делает:** не коммитит в дом, не меняет `main`, не публикует ничего
от имени основателя. Это инструмент прогона, а не голос.
